# Honeydew AI — многодорожечное обучение MIDI

Ноутбук дообучает Music Transformer из репозитория `music-composer` на многодорожечных MIDI. Результат — MIDI с отдельными инструментами и ударными, не WAV. Для первого запуска рекомендуется GPU T4 и небольшой набор MIDI.

In [ ]:
!nvidia-smi
!pip -q install pretty_midi tqdm numpy
!git clone --depth 1 --branch honeydew-multitrack https://github.com/djdhsvsbdb-design/music-composer.git /content/music-composer
%cd /content/music-composer/src
import sys, os
sys.path.insert(0, '/content/music-composer/src')


In [ ]:
!python -m unittest discover -s tests -v\n

## 1. Подключи Google Drive и положи MIDI-файлы в одну папку

Можно использовать MIDI stems из Slakh2100 Redux или собственную коллекцию. Сохраняй структуру папок: раздельные MIDI одного трека должны оставаться внутри папки исходной композиции, если они относятся к одной песне. Не загружай WAV/MP3 — этот этап обучает символьную MIDI-модель.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
MIDI_DIR = '/content/drive/MyDrive/Honeydew/MIDI'  # измени, если папка называется иначе
DATA_FILE = '/content/drive/MyDrive/Honeydew/honeydew_data.pt'
OUT_DIR = '/content/drive/MyDrive/Honeydew/checkpoints'
assert os.path.isdir(MIDI_DIR), f'Не найдена папка MIDI: {MIDI_DIR}'
print('MIDI folder:', MIDI_DIR)


## 2. Подготовь до 2 000 обучающих окон

Окна — это фрагменты композиций, а не 2 000 уникальных песен. Скрипт сохраняет ID исходного файла, чтобы train/validation можно было разделять по композициям. Для качественного результата желательно иметь сотни разных композиций.

In [ ]:
!python prepare_honeydew.py --midi_dir "$MIDI_DIR" --out "$DATA_FILE" --target_windows 2000 --seq_len 1024
import torch
data = torch.load(DATA_FILE, map_location='cpu', weights_only=False)
print('Окон:', len(data['samples']))
print('Исходных MIDI:', data['source_files'])
print('Ошибок чтения:', len(data['errors']))
assert data['source_files'] >= 2, 'Нужно минимум два исходных MIDI-файла для честной проверки.'


## 3. Обучи модель

Первая конфигурация компактнее исходной 12-слойной модели, чтобы повысить шанс запуска на Colab T4. При нехватке памяти уменьши `BATCH_SIZE` до 1. Обучение может занять значительное время; checkpoint сохраняется после каждой эпохи на Google Drive.

In [ ]:
EPOCHS = 10
BATCH_SIZE = 2
!python train_honeydew.py --data "$DATA_FILE" --out_dir "$OUT_DIR" --epochs $EPOCHS --batch_size $BATCH_SIZE --seq_len 1024


## 4. Сгенерируй MIDI и проверь результат

Открой полученный MIDI в Honeydew или другой DAW. Модель пока не управляет эффектами DAW: она генерирует партии и назначение GM-инструментов; эффекты и микширование выполняются в Honeydew.

In [ ]:
!python generate_honeydew.py --checkpoint "$OUT_DIR/best.pt" --out /content/honeydew_generated.mid --tokens 1024
from google.colab import files
files.download('/content/honeydew_generated.mid')


## Важные ограничения

- Модель обучается на MIDI-событиях и не генерирует аудиоволну.
- Качество зависит от количества и чистоты многодорожечных MIDI.
- Для полноценного продакшена потребуются тесты на разных жанрах, проверка декодера и адаптер импорта в Honeydew.
- Не смешивай разные MIDI stems из разных композиций в одну папку одного источника: это приведёт к неверному группированию при проверке.
